# YOLO Segmentation — Custom Dataset Training

This notebook covers the full pipeline:
1. Environment setup
2. Dataset preparation & validation
3. Training configuration
4. Training & monitoring
5. Evaluation & inference
6. Export

## 1. Environment Setup

In [ ]:
!pip install ultralytics supervision roboflow -q

In [ ]:
import os
import yaml
import shutil
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

from ultralytics import YOLO
import supervision as sv

print("All imports OK")

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Dataset Preparation

### Option A — Import from Roboflow

In [ ]:
USE_ROBOFLOW = False

if USE_ROBOFLOW:
    from roboflow import Roboflow

    RF_API_KEY   = "YOUR_API_KEY"
    RF_WORKSPACE = "your-workspace"
    RF_PROJECT   = "your-project"
    RF_VERSION   = 1

    rf = Roboflow(api_key=RF_API_KEY)
    project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
    dataset = project.version(RF_VERSION).download("yolov8")
    DATASET_PATH = Path(dataset.location)
else:
    DATASET_PATH = Path("dataset")

### Option B — Use a local dataset

Expected directory layout (YOLO segmentation format):
```
dataset/
  images/
    train/  *.jpg | *.png
    val/
    test/          # optional
  labels/
    train/  *.txt  # one file per image
    val/
    test/
  data.yaml
```

Each label `.txt` line: `<class_id> x1 y1 x2 y2 ... xn yn`  
All coordinates are normalised to [0, 1].

In [ ]:
DATASET_PATH = Path("dataset")

for split in ["train", "val", "test"]:
    (DATASET_PATH / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET_PATH / "labels" / split).mkdir(parents=True, exist_ok=True)

print("Dataset directories ready.")

### Create / Edit `data.yaml`

In [ ]:
CLASS_NAMES = ["class_0", "class_1", "class_2"]

data_yaml = {
    "path": str(DATASET_PATH.resolve()),
    "train": "images/train",
    "val":   "images/val",
    "test":  "images/test",
    "nc":    len(CLASS_NAMES),
    "names": CLASS_NAMES,
}

yaml_path = DATASET_PATH / "data.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"data.yaml written to {yaml_path}")
print(yaml.dump(data_yaml))

### Dataset Statistics

In [ ]:
def count_split(split):
    img_dir = DATASET_PATH / "images" / split
    lbl_dir = DATASET_PATH / "labels" / split
    imgs = list(img_dir.glob("*.[jp][pn]g")) + list(img_dir.glob("*.jpeg"))
    lbls = list(lbl_dir.glob("*.txt"))
    return len(imgs), len(lbls)

for split in ["train", "val", "test"]:
    n_img, n_lbl = count_split(split)
    print(f"{split:>6}: {n_img} images | {n_lbl} labels")

### Visualise Sample Annotations

In [ ]:
def draw_segmentation_labels(image_path, label_path, class_names, alpha=0.4):
    img = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    overlay = img.copy()
    colors = plt.cm.tab20.colors

    if not Path(label_path).exists():
        return img

    with open(label_path) as f:
        for line in f:
            parts = list(map(float, line.strip().split()))
            cls = int(parts[0])
            coords = np.array(parts[1:]).reshape(-1, 2)
            coords = (coords * [w, h]).astype(np.int32)
            color = tuple(int(c * 255) for c in colors[cls % len(colors)][:3])
            cv2.fillPoly(overlay, [coords], color)
            cv2.polylines(img, [coords], True, color, 2)
            cv2.putText(img, class_names[cls], tuple(coords[0]),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    return cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)


train_images = sorted((DATASET_PATH / "images" / "train").glob("*.[jp][pn]g"))
n_show = min(6, len(train_images))

if n_show == 0:
    print("No training images found yet.")
else:
    fig, axes = plt.subplots(2, (n_show + 1) // 2, figsize=(16, 8))
    axes = axes.flatten()
    for i, img_path in enumerate(train_images[:n_show]):
        lbl_path = DATASET_PATH / "labels" / "train" / (img_path.stem + ".txt")
        vis = draw_segmentation_labels(img_path, lbl_path, CLASS_NAMES)
        axes[i].imshow(vis)
        axes[i].set_title(img_path.name, fontsize=8)
        axes[i].axis("off")
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    plt.show()

## 3. Model & Training Configuration

In [ ]:
MODEL_SIZE = "n"     # n | s | m | l | x
YOLO_VERSION = 11    # 8 | 11

MODEL_NAME = f"yolo{YOLO_VERSION}{MODEL_SIZE}-seg.pt"
print(f"Base model: {MODEL_NAME}")

model = YOLO(MODEL_NAME)
print(model.info())

In [ ]:
TRAIN_CONFIG = dict(
    data       = str(yaml_path),
    epochs     = 100,
    imgsz      = 640,
    batch      = 16,
    patience   = 20,
    device     = 0 if torch.cuda.is_available() else "cpu",
    workers    = 4,
    project    = "runs/segment",
    name       = "custom_seg",
    exist_ok   = True,
    pretrained = True,
    optimizer  = "AdamW",
    lr0        = 1e-3,
    lrf        = 0.01,
    momentum   = 0.937,
    weight_decay = 5e-4,
    warmup_epochs = 3,
    cos_lr     = True,
    augment    = True,
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,
    degrees    = 10.0,
    translate  = 0.1,
    scale      = 0.5,
    flipud     = 0.0,
    fliplr     = 0.5,
    mosaic     = 1.0,
    mixup      = 0.1,
    copy_paste = 0.1,
    val        = True,
    save       = True,
    save_period = 10,
    plots      = True,
)

print("Training config:")
for k, v in TRAIN_CONFIG.items():
    print(f"  {k:<20}: {v}")

## 4. Training

In [ ]:
results = model.train(**TRAIN_CONFIG)

BEST_WEIGHTS = Path(results.save_dir) / "weights" / "best.pt"
LAST_WEIGHTS = Path(results.save_dir) / "weights" / "last.pt"
print(f"\nBest weights: {BEST_WEIGHTS}")

### Training Curves

In [ ]:
import pandas as pd

results_csv = Path(results.save_dir) / "results.csv"
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

metrics = [
    ("train/box_loss", "val/box_loss"),
    ("train/seg_loss", "val/seg_loss"),
    ("metrics/mAP50(M)", "metrics/mAP50-95(M)"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (m1, m2) in zip(axes, metrics):
    if m1 in df.columns:
        ax.plot(df[m1], label=m1.split("/")[-1])
    if m2 in df.columns:
        ax.plot(df[m2], label=m2.split("/")[-1], linestyle="--")
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(alpha=0.3)
plt.suptitle("Training Curves", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Evaluation

In [ ]:
best_model = YOLO(str(BEST_WEIGHTS))

metrics = best_model.val(
    data    = str(yaml_path),
    split   = "val",
    imgsz   = TRAIN_CONFIG["imgsz"],
    batch   = TRAIN_CONFIG["batch"],
    device  = TRAIN_CONFIG["device"],
    plots   = True,
)

print(f"mAP50   (mask): {metrics.seg.map50:.4f}")
print(f"mAP50-95(mask): {metrics.seg.map:.4f}")
print(f"mAP50   (box):  {metrics.box.map50:.4f}")
print(f"mAP50-95(box):  {metrics.box.map:.4f}")

In [ ]:
per_class = pd.DataFrame({
    "class": CLASS_NAMES,
    "AP50": metrics.seg.ap50,
    "AP50-95": metrics.seg.ap,
}).set_index("class")

per_class.sort_values("AP50", ascending=True).plot(
    kind="barh", figsize=(10, max(4, len(CLASS_NAMES) * 0.4 + 1)), colormap="viridis"
)
plt.title("Per-class Mask AP")
plt.xlabel("AP")
plt.axvline(0.5, color="red", linestyle="--", alpha=0.6, label="AP=0.5")
plt.legend()
plt.tight_layout()
plt.show()

per_class

## 6. Inference

In [ ]:
CONF_THRESHOLD = 0.25
IOU_THRESHOLD  = 0.45

test_images = sorted((DATASET_PATH / "images" / "test").glob("*.[jp][pn]g"))
if not test_images:
    test_images = sorted((DATASET_PATH / "images" / "val").glob("*.[jp][pn]g"))

n_show = min(4, len(test_images))

if n_show == 0:
    print("No test/val images found.")
else:
    fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 5))
    if n_show == 1:
        axes = [axes]

    for ax, img_path in zip(axes, test_images[:n_show]):
        preds = best_model.predict(
            source=str(img_path),
            conf=CONF_THRESHOLD,
            iou=IOU_THRESHOLD,
            verbose=False,
        )
        vis = preds[0].plot()
        ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
        ax.set_title(img_path.name, fontsize=8)
        ax.axis("off")

    plt.suptitle("Predictions", fontsize=13)
    plt.tight_layout()
    plt.show()

### Batch Inference on a Folder

In [ ]:
INFERENCE_SOURCE = str(DATASET_PATH / "images" / "test")
INFERENCE_OUTPUT = "inference_results"

batch_preds = best_model.predict(
    source  = INFERENCE_SOURCE,
    conf    = CONF_THRESHOLD,
    iou     = IOU_THRESHOLD,
    save    = True,
    save_txt= True,
    project = INFERENCE_OUTPUT,
    name    = "run",
    exist_ok= True,
    verbose = False,
)

print(f"Results saved to {INFERENCE_OUTPUT}/run/")

## 7. Export

In [ ]:
EXPORT_FORMAT = "onnx"   # onnx | torchscript | tflite | coreml | engine (TensorRT)

exported_path = best_model.export(
    format  = EXPORT_FORMAT,
    imgsz   = TRAIN_CONFIG["imgsz"],
    half    = False,
    dynamic = True if EXPORT_FORMAT == "onnx" else False,
    simplify= True if EXPORT_FORMAT == "onnx" else False,
)

print(f"Model exported to: {exported_path}")

### Quick ONNX Validation

In [ ]:
if EXPORT_FORMAT == "onnx":
    import onnx
    onnx_model = onnx.load(str(exported_path))
    onnx.checker.check_model(onnx_model)
    print("ONNX model is valid.")

    for inp in onnx_model.graph.input:
        shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
        print(f"Input  '{inp.name}': {shape}")
    for out in onnx_model.graph.output:
        shape = [d.dim_value for d in out.type.tensor_type.shape.dim]
        print(f"Output '{out.name}': {shape}")

## 8. Resume / Fine-tune from Checkpoint

In [ ]:
RESUME_TRAINING = False

if RESUME_TRAINING:
    resume_model = YOLO(str(LAST_WEIGHTS))
    resume_results = resume_model.train(
        resume=True,
        epochs=TRAIN_CONFIG["epochs"] + 50,
    )

In [ ]:
FINETUNE = False

if FINETUNE:
    ft_config = TRAIN_CONFIG.copy()
    ft_config.update(dict(
        name       = "custom_seg_finetune",
        epochs     = 30,
        lr0        = 1e-4,
        freeze     = 10,
    ))
    ft_model = YOLO(str(BEST_WEIGHTS))
    ft_results = ft_model.train(**ft_config)